# 04 — Hiểu và tạo các đặc trưng

Tương ứng src/features.py. Học cách biến giá/khối lượng thành 32 đầu vào và gắn nhãn của phiên tiếp theo. Notebook này minh họa; notebook 05 mới xuất bảng thực nghiệm chung.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Snapshot giá FPT; không cần mô hình đã huấn luyện.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Đọc dữ liệu và nhóm đầu vào

23 chỉ báo từ close/volume + 4 đặc trưng nến + 5 trường OHLCV hiện tại = 32 biến. Chỉ số RSI được chia 100; các cột có hậu tố ratio là tỷ lệ, không phải giá tuyệt đối.

In [ ]:
from features import load_price_data, build_feature_table, FEATURE_COLUMNS, OHLC_FEATURE_COLUMNS, RAW_PRICE_FEATURE_COLUMNS, PRICE_ONLY_FEATURE_COLUMNS
prices = load_price_data(PRICE_PATH)
display(pd.DataFrame([
    {"Nhóm": "Chỉ báo close/volume", "Số biến": len(FEATURE_COLUMNS), "Tên biến": ", ".join(FEATURE_COLUMNS)},
    {"Nhóm": "Nến OHLC", "Số biến": len(OHLC_FEATURE_COLUMNS), "Tên biến": ", ".join(OHLC_FEATURE_COLUMNS)},
    {"Nhóm": "OHLCV gốc hiện tại", "Số biến": len(RAW_PRICE_FEATURE_COLUMNS), "Tên biến": ", ".join(RAW_PRICE_FEATURE_COLUMNS)},
]))

## 5. Tính nhẩm một nhóm chỉ báo

Bảng 5 giá chỉ đủ minh họa Return và SMA3; không đủ để tính các chỉ báo 14/20/50 phiên của mô hình. SMA3 là trung bình 3 giá gần nhất. SMA ratio = giá hiện tại / SMA3 − 1.

In [ ]:
mini = pd.DataFrame({"close": [10.0, 12.0, 11.0, 14.0, 13.0]})
mini["return_1d"] = mini["close"].pct_change()
mini["sma_3"] = mini["close"].rolling(3).mean()
mini["sma_3_ratio"] = mini["close"] / mini["sma_3"] - 1.0
display(mini)
print("Ngày 2: 12 / 10 - 1 =", 12 / 10 - 1, "= 20%")
print("Ngày 3: SMA3 = (10 + 12 + 11) / 3 =", (10 + 12 + 11) / 3)

## 6. Tạo bảng đặc trưng thật

Chỉ dùng cửa sổ quá khứ. Các dòng đầu thiếu lịch sử cho chỉ báo dài bị bỏ; đây là giai đoạn chờ đủ lịch sử, không lấy giá tương lai điền vào.

In [ ]:
features = build_feature_table(prices)
print("Dòng giá gốc:", len(prices), "| Dòng có chỉ báo:", len(features))
display(features[["date", "current_close", "return_1d", "sma_5_ratio", "rsi_14", "candle_range_pct", "target_date", "next_close", "direction"]].tail(8))

## 7. Theo dõi một ngày giao dịch có thật

Chọn ngày có sẵn trước phiên cuối. Return hôm đó dùng phiên trước; next_close là nhãn của phiên sau, chỉ dùng để huấn luyện/chấm điểm.

In [ ]:
example_date = features.loc[features["target_date"].notna(), "date"].iloc[-1]
row = features.loc[features["date"].eq(example_date)].iloc[0]
raw_index = prices.index[prices["time"].eq(example_date)][0]
previous_close = float(prices.loc[raw_index - 1, "close"])
display(pd.Series({
    "Ngày đầu vào t": row["date"], "Close phiên trước": previous_close,
    "Close t": row["current_close"], "Return đã biết": row["current_close"] / previous_close - 1,
    "Phiên nhãn t+1": row["target_date"], "Close t+1 (nhãn)": row["next_close"],
    "Nhãn 1=Tăng, 0=Giảm": row["direction"],
}).to_frame("Một mẫu thật"))

## 8. Tách X khỏi nhãn y

X là những điều đã biết khi đóng cửa t; y là đáp án phiên sau. Không cho next_close, next_return, direction hoặc target_date vào X. Dòng cuối có X nhưng chưa có đáp án, giữ để dự báo.

In [ ]:
X = features[PRICE_ONLY_FEATURE_COLUMNS]
forbidden_targets = {"next_close", "next_return", "direction", "target_date"}
assert not forbidden_targets.intersection(X.columns)
display(X.tail())
display(features[["date", "target_date", "next_close", "direction"]].tail())